# 14 — Silver Validation

**Objetivo:** ejecutar validaciones cruzadas sobre todos los artefactos Silver construidos en 10–13: artifacts, lineage, schema version, grains, FK coverage y consistencia de entity resolution.

In [1]:
import pandas as pd
from argos.common.paths import ProjectPaths
from argos.config.silver import load_silver_spec
from argos.silver.validation import BUILD_IMPLEMENTATION

paths = ProjectPaths.discover(); paths.ensure_runtime_dirs()
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")

for report_name in ["10_validation.parquet", "11_validation.parquet", "12_validation.parquet", "13_validation.parquet"]:
    frame = pd.read_parquet(paths.reports / "silver" / report_name)
    blocking = frame.loc[frame["blocking"]]
    assert not blocking.empty and blocking["passed"].all(), f"Prerequisite failed: {report_name}"

{"implementation": BUILD_IMPLEMENTATION, "schema_version": SILVER_SPEC.schema_version}

{'implementation': 'm2-14-cross-silver-validation-v2',
 'schema_version': 'm2-silver-v1'}

In [2]:
from argos.silver.validation import validate_silver_m2

result = validate_silver_m2(
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    project_config_path=paths.configs / "project.toml",
    silver_spec=SILVER_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
)
result.validation

,check_id,category,blocking,passed,observed,requirement
0,silver_artifacts_complete,artifacts,True,True,"{""expected_count"": 12, ""missing"": []}",all canonical M2 Silver artifacts from noteboo...
1,component_manifests_complete,lineage,True,True,"{""missing"": []}",notebooks 10–13 persist their lineage manifests
2,component_blocking_checks,component_validation,True,True,"{""10_validation.parquet"": {""blocking"": 10, ""ok...",every blocking validation from notebooks 10–13...
3,silver_schema_version_consistent,lineage,True,True,"{""addition_event_versions.parquet"": [""m2-silve...",every Silver artifact declares schema m2-silve...
4,contract_supplier_cross_coverage,relationships,True,True,"{""contracts"": 6066730, ""coverage"": 0.946646545...",canonical contract→supplier coverage >= 90% wi...
5,bidder_entity_map_complete,entity_resolution,True,True,"{""distinct_link_ids"": 2281440, ""link_rows"": 22...",every bidder-process link has exactly one bidd...
6,event_contract_cross_integrity,relationships,True,True,"{""contract_coverage"": 0.9999182554934971, ""dis...",addition event grain remains unique and contra...
7,deferred_sources_remain_explicit,scope,True,True,"{""deferred_datasets"": [""payments"", ""sanctions""...",payments and sanctions remain explicit deferre...


## Issue register

`m2_issue_register.parquet` contiene únicamente checks que no pasaron. Un warning puede sobrevivir M2 si está explícitamente documentado; un issue `blocking` no.

In [3]:
result.issues

,check_id,category,severity,observed,requirement,action


In [4]:
blocking = result.validation.loc[result.validation["blocking"]]
failed = blocking.loc[~blocking["passed"], "check_id"].tolist()
print("14 ✅ PASS" if result.ready_for_gate else "14 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
if failed:
    print("Failed:", ", ".join(failed))
assert result.ready_for_gate, "Silver cross-validation remains OPEN; resolve blocking checks before M2 gate."

14 ✅ PASS
Blocking checks: 8/8
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/silver_m2_validation.json


**Siguiente:** `15_m2_gate.ipynb`.